# Task Overview
1. Implement hypothesis function in vector form
2. Implement loss function (MSE) in vector form
3. Implement one step of gradient descent
4. Find optimal parameters for house price prediction
5. Find parameters using analytical solution
6. Compare with scikit-learn's LinearRegression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 1. Hypothesis Function (Vector Form)

In [ ]:
def hypothesis(X, w):
    return X @ w

X_test = np.array([[1, 1000, 2, 3],
                   [1, 1500, 3, 4]])
w_test = np.array([0.5, 0.1, 0.2, 0.3])
print(f"Hypothesis test result: {hypothesis(X_test, w_test)}")

## 2. Loss Function (Vector Form)

In [ ]:
def loss_function(X, y, w):
    m = len(y)
    error = hypothesis(X, w) - y
    return (1 / (2 * m)) * (error @ error)

y_test = np.array([250000, 350000])
print(f"Loss function test result: {loss_function(X_test, y_test, w_test):.2f}")

## 3. Gradient Descent Step

In [ ]:
def gradient_descent_step(X, y, w, alpha):
    m = len(y)
    error = hypothesis(X, w) - y
    gradient = (1 / m) * (X.T @ error)
    w_updated = w - alpha * gradient
    return w_updated, gradient

w_updated_test, grad_test = gradient_descent_step(X_test, y_test, w_test, alpha=0.0001)
print(f"Updated weights: {w_updated_test}")
print(f"Gradient: {grad_test}")

## 4. Finding Optimal Parameters Using Gradient Descent

In [ ]:
np.random.seed(42)
m = 100 

area = np.random.uniform(800, 4000, m)
bathrooms = np.random.randint(1, 5, m)
bedrooms = np.random.randint(1, 6, m)

price = (50000 + 
         150 * area + 
         10000 * bathrooms + 
         5000 * bedrooms + 
         np.random.normal(0, 20000, m))

X = np.column_stack([np.ones(m), area, bathrooms, bedrooms])
y = price

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"\nFirst 5 samples:")
print(f"Area: {area[:5]}")
print(f"Bathrooms: {bathrooms[:5]}")
print(f"Bedrooms: {bedrooms[:5]}")
print(f"Price: {y[:5]}")

In [ ]:
scaler = StandardScaler()
X_scaled = np.column_stack([np.ones(m), scaler.fit_transform(X[:, 1:])])

print("Features scaled successfully")
print(f"Scaled features mean: {X_scaled[:, 1:].mean(axis=0)}")
print(f"Scaled features std: {X_scaled[:, 1:].std(axis=0)}")

In [ ]:
def train_gradient_descent(X, y, alpha=0.1, iterations=10000):
    n = X.shape[1]
    w = np.zeros(n)
    loss_history = []
    
    for i in range(iterations):
        w, gradient = gradient_descent_step(X, y, w, alpha)
        loss = loss_function(X, y, w)
        loss_history.append(loss)
        
        if i % 1000 == 0:
            print(f"Iteration {i}, Loss: {loss:.2f}")
    
    return w, loss_history

print("Training with Gradient Descent...\n")
w_gd, loss_history = train_gradient_descent(X_scaled, y, alpha=0.1, iterations=10000)

print(f"\nOptimal weights (Gradient Descent): {w_gd}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(loss_history)
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.title('Loss Convergence During Gradient Descent')
plt.grid(True)
plt.show()

## 5. Analytical Solution (Normal Equation)

In [ ]:
def analytical_solution(X, y):
    return np.linalg.inv(X.T @ X) @ X.T @ y

w_analytical = analytical_solution(X_scaled, y)
print(f"Optimal weights (Analytical Solution): {w_analytical}")

print(f"\nComparison:")
print(f"Gradient Descent:  {w_gd}")
print(f"Analytical:        {w_analytical}")
print(f"Difference:        {np.abs(w_gd - w_analytical)}")

## 6. Comparison with Scikit-Learn's LinearRegression

In [ ]:
X_features = X_scaled[:, 1:]  

lr = LinearRegression()
lr.fit(X_features, y)

w_sklearn = np.concatenate([[lr.intercept_], lr.coef_])
print(f"Optimal weights (Scikit-Learn): {w_sklearn}")

print(f"\n{'='*60}")
print(f"{'Method':<25} {'Weights':<40}")
print(f"{'='*60}")
print(f"{'Gradient Descent':<25} {str(np.round(w_gd, 2)):<40}")
print(f"{'Analytical Solution':<25} {str(np.round(w_analytical, 2)):<40}")
print(f"{'Scikit-Learn':<25} {str(np.round(w_sklearn, 2)):<40}")
print(f"{'='*60}")

In [ ]:
y_pred_gd = hypothesis(X_scaled, w_gd)
y_pred_analytical = hypothesis(X_scaled, w_analytical)
y_pred_sklearn = lr.predict(X_features)

mse_gd = np.mean((y - y_pred_gd) ** 2)
mse_analytical = np.mean((y - y_pred_analytical) ** 2)
mse_sklearn = np.mean((y - y_pred_sklearn) ** 2)

print(f"Mean Squared Error Comparison:")
print(f"Gradient Descent:  {mse_gd:.2f}")
print(f"Analytical:        {mse_analytical:.2f}")
print(f"Scikit-Learn:      {mse_sklearn:.2f}")

r2_gd = 1 - (np.sum((y - y_pred_gd) ** 2) / np.sum((y - np.mean(y)) ** 2))
r2_analytical = 1 - (np.sum((y - y_pred_analytical) ** 2) / np.sum((y - np.mean(y)) ** 2))
r2_sklearn = lr.score(X_features, y)

print(f"\nR-squared Comparison:")
print(f"Gradient Descent:  {r2_gd:.4f}")
print(f"Analytical:        {r2_analytical:.4f}")
print(f"Scikit-Learn:      {r2_sklearn:.4f}")

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
area_range = np.linspace(800, 4000, 100)
avg_bathrooms = np.mean(bathrooms)
avg_bedrooms = np.mean(bedrooms)

X_plot = np.column_stack([
    np.ones(100),
    area_range,
    np.full(100, avg_bathrooms),
    np.full(100, avg_bedrooms)
])
X_plot_scaled = np.column_stack([np.ones(100), scaler.transform(X_plot[:, 1:])])

y_pred_gd_plot = hypothesis(X_plot_scaled, w_gd)
y_pred_analytical_plot = hypothesis(X_plot_scaled, w_analytical)
y_pred_sklearn_plot = lr.predict(X_plot_scaled[:, 1:])

plt.scatter(area, y, alpha=0.3, label='Actual Data', color='blue')
plt.plot(area_range, y_pred_gd_plot, label='Gradient Descent', linewidth=2, linestyle='--')
plt.plot(area_range, y_pred_analytical_plot, label='Analytical', linewidth=2, linestyle='-.')
plt.plot(area_range, y_pred_sklearn_plot, label='Scikit-Learn', linewidth=2, linestyle=':')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price (USD)')
plt.title('House Price Predictions vs Area\n(avg bathrooms={}, avg bedrooms={})'.format(
    round(avg_bathrooms, 1), round(avg_bedrooms, 1)))
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
feature_names = ['Bias', 'Area', 'Bathrooms', 'Bedrooms']
x_pos = np.arange(len(feature_names))
width = 0.25

w_gd_norm = w_gd / np.max(np.abs(w_gd))
w_analytical_norm = w_analytical / np.max(np.abs(w_analytical))
w_sklearn_norm = w_sklearn / np.max(np.abs(w_sklearn))

plt.bar(x_pos - width, w_gd_norm, width, label='Gradient Descent', alpha=0.8)
plt.bar(x_pos, w_analytical_norm, width, label='Analytical', alpha=0.8)
plt.bar(x_pos + width, w_sklearn_norm, width, label='Scikit-Learn', alpha=0.8)
plt.xlabel('Features')
plt.ylabel('Normalized Weights')
plt.title('Comparison of Model Weights (Normalized)')
plt.xticks(x_pos, feature_names)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Making Predictions

In [ ]:
new_house = np.array([[1, 2000, 3, 4]])
new_house_scaled = np.column_stack([np.ones(1), scaler.transform(new_house[:, 1:])])

pred_gd = hypothesis(new_house_scaled, w_gd)[0]
pred_analytical = hypothesis(new_house_scaled, w_analytical)[0]
pred_sklearn = lr.predict(new_house_scaled[:, 1:])[0]

print(f"House Features: Area=2000 sq ft, Bathrooms=3, Bedrooms=4\n")
print(f"Predicted Prices:")
print(f"Gradient Descent:  ${pred_gd:,.2f}")
print(f"Analytical:        ${pred_analytical:,.2f}")
print(f"Scikit-Learn:      ${pred_sklearn:,.2f}")
print(f"\nDifference (max):  ${abs(pred_gd - pred_sklearn):,.2f}")